In [ ]:
import json
import csv
import subprocess
import numpy as np
import pandas as pd
from pathlib import Path

results_file = Path("./outputs/results.csv")

tasks_setup = {
    "rte":  {"lr": 1e-3, "metric": "Accuracy"},
    "mrpc": {"lr": 7e-4, "metric": "F1"},
    "cola": {"lr": 7e-4, "metric": "MCC"},
    "qnli": {"lr": 1e-4, "metric": "Accuracy"}
}
#  bert fullft: rte 2e-5
#  bert fullt: cola 2e-5
#  bert bitfit: cola 7e-4

#  dbert fullft: qnli 2e-4
#  dbert bitfit: qnli 5e-4
#  dbert bitfit: rte 3e-4
#  dbert fullft: rte 3e-5
#  dbert fullft: mrpc 1e-5

In [ ]:
def run_task(task, 
             fine_tune_type="bitfit", 
             model="bert-base-cased", 
             seeds=[1, 2, 3, 4, 5], 
             batch=16, 
             gpu=0,
             lr="2e-5",
             epoch="20"):
             
    # On expérimente / reproduit que sur BERT Base et DBert ici
    model_name = "bb" if model=="bert-base-cased" else "d-bert"

    config = tasks_setup[task]
    metric = config["metric"]
    results = []

    for seed in seeds:
        output_path = Path("./outputs") / f"{model_name}_{fine_tune_type}_{task}_seed{seed}"
        output_path.mkdir(parents=True, exist_ok=True)

        subprocess.run(["python", "run_glue.py",
                        "--output-path", str(output_path),
                        "--task-name", task,
                        "--model-name", model,
                        "--fine-tune-type", fine_tune_type,
                        "--learning-rate", str(lr),
                        "--epochs", str(epoch),
                        "--batch-size", str(batch),
                        "--gpu-device", str(gpu),
                        "--seed", str(seed),
                        "--verbose"])

        with open(output_path / "eval_results.json") as f:
            acc = json.load(f)["validation"][metric]
        results.append(acc)

    mean = np.mean(results) * 100
    std = np.std(results) * 100

    results_file.parent.mkdir(parents=True, exist_ok=True)
    write_header = not results_file.exists()
    with open(results_file, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["model", "task", "fine_tune_type", "metric", "mean", "std"])
        if write_header:
            writer.writeheader()
        writer.writerow({"model": model, 
                         "task": task, 
                         "fine_tune_type": fine_tune_type,
                         "metric": metric, 
                         "mean": mean, 
                         "std": std})

In [ ]:
!pip install torch transformers datasets tokenizers scikit-learn scipy sentencepiece sacremoses tqdm matplotlib seaborn pandas numpy

In [15]:
run_task("mrpc", fine_tune_type="full_ft", model="distilbert-base-cased", lr=1e-5, epoch=8)

2026-03-19 00:23:57,260 - run_glue - INFO - ############################################################################################
2026-03-19 00:23:57,260 - run_glue - INFO - ############################################################################################
2026-03-19 00:23:57,261 - run_glue - INFO - ############################################################################################
2026-03-19 00:23:57,261 - run_glue - INFO - 
2026-03-19 00:23:57,261 - run_glue - INFO - Training Details: 
2026-03-19 00:23:57,261 - run_glue - INFO - ----------------------------------------------
2026-03-19 00:23:57,261 - run_glue - INFO - Model Name: distilbert-base-cased
2026-03-19 00:23:57,261 - run_glue - INFO - Task Name: mrpc
2026-03-19 00:23:57,261 - run_glue - INFO - Fine Tuning Type: full_ft
2026-03-19 00:23:57,261 - run_glue - INFO - Output Directory: outputs/d-bert_full_ft_mrpc_seed1
2026-03-19 00:23:57,261 - run_glue - INFO - Running on GPU #0
2026-03-19 00:23:57,261 

2026-03-19 00:23:57,699 - _client - INFO - HTTP Request: HEAD https://huggingface.co/datasets/nyu-mll/glue/resolve/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c/glue.py "HTTP/1.1 404 Not Found"
2026-03-19 00:23:57,994 - _client - INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/glue/glue.py "HTTP/1.1 200 OK"
2026-03-19 00:23:58,135 - _client - INFO - HTTP Request: GET https://huggingface.co/api/datasets/glue/revision/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c "HTTP/1.1 307 Temporary Redirect"
2026-03-19 00:23:58,320 - _client - INFO - HTTP Request: GET https://huggingface.co/api/datasets/nyu-mll/glue/revision/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c "HTTP/1.1 200 OK"
2026-03-19 00:23:58,421 - _client - INFO - HTTP Request: HEAD https://huggingface.co/datasets/glue/resolve/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c/.huggingface.yaml "HTTP/1.1 307 Temporary Redirect"
2026-03-19 00:23:58,521 - _client - INFO - HTTP Request: HEAD https://huggingface.co/da

Map: 7336 examples [00:00, 11540.00 examples/s]        
Map: 816 examples [00:00, 8459.94 examples/s]        
Map: 3450 examples [00:00, 11972.42 examples/s]        


2026-03-19 00:24:02,424 - _client - INFO - HTTP Request: HEAD https://huggingface.co/distilbert-base-cased/resolve/main/config.json "HTTP/1.1 200 OK"


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3567.80it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

2026-03-19 01:09:36,012 - _client - INFO - HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-cased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-03-19 01:09:36,112 - _client - INFO - HTTP Request: GET https://huggingface.co/api/models/distilbert-base-cased/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-03-19 01:09:36,215 - _client - INFO - HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-cased/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"


Map: 7336 examples [00:00, 10721.92 examples/s]        
Map: 816 examples [00:00, 7156.32 examples/s]        
Map: 3450 examples [00:00, 9816.18 examples/s]         


2026-03-19 01:09:38,732 - _client - INFO - HTTP Request: HEAD https://huggingface.co/distilbert-base-cased/resolve/main/config.json "HTTP/1.1 200 OK"


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3267.38it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.




Trainable Components:
----------------------------------------

distilbert.embeddings.word_embeddings.weight   --->   torch.Size([28996, 768])
distilbert.embeddings.position_embeddings.weight   --->   torch.Size([512, 768])
distilbert.embeddings.LayerNorm.weight   --->   torch.Size([768])
distilbert.embeddings.LayerNorm.bias   --->   torch.Size([768])
distilbert.transformer.layer.0.attention.q_lin.weight   --->   torch.Size([768, 768])
distilbert.transformer.layer.0.attention.q_lin.bias   --->   torch.Size([768])
distilbert.transformer.layer.0.attention.k_lin.weight   --->   torch.Size([768, 768])
distilbert.transformer.layer.0.attention.k_lin.bias   --->   torch.Size([768])
distilbert.transformer.layer.0.attention.v_lin.weight   --->   torch.Size([768, 768])
distilbert.transformer.layer.0.attention.v_lin.bias   --->   torch.Size([768])
distilbert.transformer.layer.0.attention.out_lin.weight   --->   torch.Size([768, 768])
distilbert.transformer.layer.0.attention.out_lin.bias   ---> 

In [16]:
run_task("mrpc", fine_tune_type="bitfit", model="distilbert-base-cased", lr=5e-4, epoch=8)

2026-03-19 01:20:57,685 - run_glue - INFO - ############################################################################################
2026-03-19 01:20:57,685 - run_glue - INFO - ############################################################################################
2026-03-19 01:20:57,685 - run_glue - INFO - ############################################################################################
2026-03-19 01:20:57,686 - run_glue - INFO - 
2026-03-19 01:20:57,686 - run_glue - INFO - Training Details: 
2026-03-19 01:20:57,686 - run_glue - INFO - ----------------------------------------------
2026-03-19 01:20:57,686 - run_glue - INFO - Model Name: distilbert-base-cased
2026-03-19 01:20:57,686 - run_glue - INFO - Task Name: mrpc
2026-03-19 01:20:57,686 - run_glue - INFO - Fine Tuning Type: bitfit
2026-03-19 01:20:57,686 - run_glue - INFO - Output Directory: outputs/d-bert_bitfit_mrpc_seed1
2026-03-19 01:20:57,686 - run_glue - INFO - Running on GPU #0
2026-03-19 01:20:57,686 - 

2026-03-19 01:20:58,131 - _client - INFO - HTTP Request: HEAD https://huggingface.co/datasets/nyu-mll/glue/resolve/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c/glue.py "HTTP/1.1 404 Not Found"
2026-03-19 01:20:58,132 - _http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-03-19 01:20:58,419 - _client - INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/glue/glue.py "HTTP/1.1 200 OK"
2026-03-19 01:20:58,566 - _client - INFO - HTTP Request: GET https://huggingface.co/api/datasets/glue/revision/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c "HTTP/1.1 307 Temporary Redirect"
2026-03-19 01:20:58,664 - _client - INFO - HTTP Request: GET https://huggingface.co/api/datasets/nyu-mll/glue/revision/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c "HTTP/1.1 200 OK"
2026-03-19 01:20:58,765 - _client - INFO - HTTP Request: HEAD https://huggingface.co/datasets/glue/resolv

Map: 7336 examples [00:00, 11365.63 examples/s]        
Map: 816 examples [00:00, 8723.43 examples/s]        
Map: 3450 examples [00:00, 12652.29 examples/s]        


2026-03-19 01:21:02,733 - _client - INFO - HTTP Request: HEAD https://huggingface.co/distilbert-base-cased/resolve/main/config.json "HTTP/1.1 200 OK"


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4671.50it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Map: 7336 examples [00:00, 11390.07 examples/s]        
Map: 816 examples [00:00, 10607.57 examples/s]       
Map: 3450 examples [00:00, 11606.96 examples/s]        


2026-03-19 01:54:09,319 - _client - INFO - HTTP Request: HEAD https://huggingface.co/distilbert-base-cased/resolve/main/config.json "HTTP/1.1 200 OK"


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4162.05it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.




Trainable Components:
----------------------------------------

distilbert.embeddings.LayerNorm.bias   --->   torch.Size([768])
distilbert.transformer.layer.0.attention.q_lin.bias   --->   torch.Size([768])
distilbert.transformer.layer.0.attention.k_lin.bias   --->   torch.Size([768])
distilbert.transformer.layer.0.attention.v_lin.bias   --->   torch.Size([768])
distilbert.transformer.layer.0.attention.out_lin.bias   --->   torch.Size([768])
distilbert.transformer.layer.0.sa_layer_norm.bias   --->   torch.Size([768])
distilbert.transformer.layer.0.ffn.lin1.bias   --->   torch.Size([3072])
distilbert.transformer.layer.0.ffn.lin2.bias   --->   torch.Size([768])
distilbert.transformer.layer.0.output_layer_norm.bias   --->   torch.Size([768])
distilbert.transformer.layer.1.attention.q_lin.bias   --->   torch.Size([768])
distilbert.transformer.layer.1.attention.k_lin.bias   --->   torch.Size([768])
distilbert.transformer.layer.1.attention.v_lin.bias   --->   torch.Size([768])
distilbert.t

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4163.00it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
run_task("mrpc", fine_tune_type="full_ft", model="bert-base-cased", lr=2e-5, epoch=8)

2026-03-19 02:02:20,567 - run_glue - INFO - ############################################################################################
2026-03-19 02:02:20,567 - run_glue - INFO - ############################################################################################
2026-03-19 02:02:20,567 - run_glue - INFO - ############################################################################################
2026-03-19 02:02:20,567 - run_glue - INFO - 
2026-03-19 02:02:20,567 - run_glue - INFO - Training Details: 
2026-03-19 02:02:20,567 - run_glue - INFO - ----------------------------------------------
2026-03-19 02:02:20,567 - run_glue - INFO - Model Name: bert-base-cased
2026-03-19 02:02:20,567 - run_glue - INFO - Task Name: mrpc
2026-03-19 02:02:20,567 - run_glue - INFO - Fine Tuning Type: full_ft
2026-03-19 02:02:20,567 - run_glue - INFO - Output Directory: outputs/bb_full_ft_mrpc_seed1
2026-03-19 02:02:20,567 - run_glue - INFO - Running on GPU #0
2026-03-19 02:02:20,567 - run_glue

2026-03-19 02:02:21,053 - _client - INFO - HTTP Request: HEAD https://huggingface.co/datasets/nyu-mll/glue/resolve/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c/glue.py "HTTP/1.1 404 Not Found"
2026-03-19 02:02:21,353 - _client - INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/glue/glue.py "HTTP/1.1 200 OK"
2026-03-19 02:02:21,495 - _client - INFO - HTTP Request: GET https://huggingface.co/api/datasets/glue/revision/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c "HTTP/1.1 307 Temporary Redirect"
2026-03-19 02:02:21,594 - _client - INFO - HTTP Request: GET https://huggingface.co/api/datasets/nyu-mll/glue/revision/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c "HTTP/1.1 200 OK"
2026-03-19 02:02:21,695 - _client - INFO - HTTP Request: HEAD https://huggingface.co/datasets/glue/resolve/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c/.huggingface.yaml "HTTP/1.1 307 Temporary Redirect"
2026-03-19 02:02:21,797 - _client - INFO - HTTP Request: HEAD https://huggingface.co/da

Map: 7336 examples [00:00, 11329.35 examples/s]        
Map: 816 examples [00:00, 9205.31 examples/s]        
Map: 3450 examples [00:00, 10741.49 examples/s]        


2026-03-19 02:02:25,873 - _client - INFO - HTTP Request: HEAD https://huggingface.co/bert-base-cased/resolve/main/config.json "HTTP/1.1 200 OK"


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3133.24it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider trai

...

2026-03-19 03:31:08,425 - _client - INFO - HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-cased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-03-19 03:31:08,522 - _client - INFO - HTTP Request: GET https://huggingface.co/api/models/bert-base-cased/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-03-19 03:31:08,621 - _client - INFO - HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-cased/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"


Map: 7336 examples [00:00, 9531.60 examples/s]         
Map: 816 examples [00:00, 6984.14 examples/s]        
Map: 3450 examples [00:00, 11680.68 examples/s]        


2026-03-19 03:31:11,237 - _client - INFO - HTTP Request: HEAD https://huggingface.co/bert-base-cased/resolve/main/config.json "HTTP/1.1 200 OK"


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3859.77it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider trai



Trainable Components:
----------------------------------------

bert.embeddings.word_embeddings.weight   --->   torch.Size([28996, 768])
bert.embeddings.position_embeddings.weight   --->   torch.Size([512, 768])
bert.embeddings.token_type_embeddings.weight   --->   torch.Size([2, 768])
bert.embeddings.LayerNorm.weight   --->   torch.Size([768])
bert.embeddings.LayerNorm.bias   --->   torch.Size([768])
bert.encoder.layer.0.attention.self.query.weight   --->   torch.Size([768, 768])
bert.encoder.layer.0.attention.self.query.bias   --->   torch.Size([768])
bert.encoder.layer.0.attention.self.key.weight   --->   torch.Size([768, 768])
bert.encoder.layer.0.attention.self.key.bias   --->   torch.Size([768])
bert.encoder.layer.0.attention.self.value.weight   --->   torch.Size([768, 768])
bert.encoder.layer.0.attention.self.value.bias   --->   torch.Size([768])
bert.encoder.layer.0.attention.output.dense.weight   --->   torch.Size([768, 768])
bert.encoder.layer.0.attention.output.dense.bias 